# Nonlinear static analysis

A laminated cylinder under axial compression, solved with the full
Newton-Raphson method of `structsolve.Analysis`, whose callables assemble
the internal forces of `update_fint` and the tangent stiffness matrix of
`update_KC0`, `update_KCNL` and `update_KG`. The load is applied in five
increments up to 50 kN, and
the geometrically nonlinear response is compared with the linear one. The
same model, in one increment, is checked by the test
`tests/test_nonlinear_axial_compression.py`, whose class `Callables` is
used here; [Nonlinear static analysis with structsolve](ex_structsolve_nonlinear.ipynb)
explains the callables.

**Theory.** Under load control the residual
$\boldsymbol{R} = \boldsymbol{F}_{int}(\boldsymbol{u}) - \lambda
\boldsymbol{F}_{ext}$ is solved by the iterations, see
[Analyses](../analyses.rst),

$$
\boldsymbol{K}_T(\boldsymbol{u}^{(i)}) \Delta \boldsymbol{u} =
-\boldsymbol{R}(\boldsymbol{u}^{(i)}, \lambda), \qquad
\boldsymbol{u}^{(i+1)} = \boldsymbol{u}^{(i)} + \Delta \boldsymbol{u}
$$

with $\boldsymbol{K}_T = \boldsymbol{K}_{C0} + \boldsymbol{K}_{CNL} +
\boldsymbol{K}_{GNL} + \boldsymbol{K}_G$, the exact Jacobian of
$\boldsymbol{F}_{int}$, computed with the current displacements, which
gives quadratic convergence near the solution. The convergence criterion
is that of Crisfield [Crisfield1981].

In [1]:
import os
import sys
import numpy as np
from numpy import isclose, pi
import matplotlib.pyplot as plt
from scipy.sparse import coo_matrix
from scipy.sparse.linalg import spsolve
from composites import laminated_plate
from structsolve import Analysis

from bfsccylinder import (BFSCCylinder, update_KC0, DOF, DOUBLE, INT,
                          KC0_SPARSE_SIZE, KCNL_SPARSE_SIZE, KG_SPARSE_SIZE)
from bfsccylinder.quadrature import get_points_weights
from bfsccylinder.utils import assign_constant_ABD

# the callables of structsolve.Analysis, shared with the tests
sys.path.insert(0, os.path.join('..', 'tests'))
from test_nonlinear_axial_compression import Callables

from IPython.display import Markdown


def table(rows, fmt='{:.5g}'):
    """Markdown table of a list of dicts with the same keys"""
    keys = list(rows[0])
    cell = lambda v: fmt.format(v) if isinstance(v, float) else str(v)
    lines = ['| ' + ' | '.join(keys) + ' |', '|' + '---|'*len(keys)]
    lines += ['| ' + ' | '.join(cell(r[k]) for k in keys) + ' |' for r in rows]
    return Markdown('\n'.join(lines))

## Cylinder, laminate and mesh

The geometry of the cylinder of Arbocz and Starnes [ArboczStarnes2002],
$R = 203.186$ mm and $L = 355.6$ mm, with the laminate
$[45, -45, 0, 90]_s$ of total thickness 1.01539 mm and the ply constants
below. The mesh has 64 nodes around the circumference and an odd number
along the axis, such that a circle of nodes lies at $x = L/2$.

In [2]:
L = 0.3556  # m
R = 0.20318603  # m
b = 2*pi*R  # m
E11, E22, G12, nu12 = 127.629e9, 11.3074e9, 6.00257e9, 0.3002
stack = [45, -45, 0, 90, 90, 0, -45, 45]
h = 0.00101539  # m
prop = laminated_plate(stack=stack, plyt=h/len(stack),
                       laminaprop=(E11, E22, nu12, G12, G12, G12))

ny = 64
nx = int(1.5*ny*L/b)
if nx % 2 == 0:
    nx += 1

# node numbers on a grid of nx by ny + 1, the last column closing the cylinder
nids = 1 + np.arange(nx*(ny+1))
nids_mesh = nids.reshape(nx, ny+1)
nids_mesh[:, -1] = nids_mesh[:, 0]
nids = np.unique(nids_mesh)
nid_pos = dict(zip(nids, np.arange(len(nids))))

# coordinates of the nodes on the developed mid-surface
xlin = np.linspace(0, L, nx)
ylin = np.linspace(0, b, ny+1)[:-1]
xmesh, ymesh = np.meshgrid(xlin, ylin, indexing='ij')
x = xmesh.flatten()
y = ymesh.flatten()

# the four nodes of each element, counterclockwise from (x_i, y_j)
n1s = nids_mesh[:-1, :-1].flatten()
n2s = nids_mesh[1:, :-1].flatten()
n3s = nids_mesh[1:, 1:].flatten()
n4s = nids_mesh[:-1, 1:].flatten()
num_elements = len(n1s)
N = DOF*nx*ny
print('nodes:', nx*ny, ' elements:', num_elements, ' degrees of freedom:', N)

nodes: 1728  elements: 1664  degrees of freedom: 17280


## Elements and linear stiffness matrix

Each element gets the positions of its entries in the sparse arrays of
$\boldsymbol{K}_{C0}$, $\boldsymbol{K}_{CNL}$ and $\boldsymbol{K}_G$.

In [3]:
nint = 4
points, weights = get_points_weights(nint=nint)
elements = []
for k, (n1, n2, n3, n4) in enumerate(zip(n1s, n2s, n3s, n4s)):
    shell = BFSCCylinder(nint)
    shell.c1 = DOF*nid_pos[n1]
    shell.c2 = DOF*nid_pos[n2]
    shell.c3 = DOF*nid_pos[n3]
    shell.c4 = DOF*nid_pos[n4]
    shell.R = R
    shell.lex = L/(nx - 1)
    shell.ley = b/ny
    assign_constant_ABD(shell, prop)
    shell.init_k_KC0 = k*KC0_SPARSE_SIZE
    shell.init_k_KCNL = k*KCNL_SPARSE_SIZE
    shell.init_k_KG = k*KG_SPARSE_SIZE
    elements.append(shell)

Kr = np.zeros(KC0_SPARSE_SIZE*num_elements, dtype=INT)
Kc = np.zeros(KC0_SPARSE_SIZE*num_elements, dtype=INT)
Kv = np.zeros(KC0_SPARSE_SIZE*num_elements, dtype=DOUBLE)
for shell in elements:
    update_KC0(shell, points, weights, Kr, Kc, Kv)
KC0 = coo_matrix((Kv, (Kr, Kc)), shape=(N, N)).tocsc()

## Boundary conditions and load

$v$ and $w$ are constrained at the nodes of both ends and $u$ at the nodes
of the middle section $x = L/2$, which removes the axial rigid-body
translation and leaves the ends free to move axially. The load of 50 kN is
distributed over the nodes of both ends at the positions of $u$, with
opposite signs: a self-equilibrated compression. All prescribed values are
zero, which the expansion `u[bu] = c` of the callables assumes.

In [4]:
bk = np.zeros(N, dtype=bool)
ends = isclose(x, 0) | isclose(x, L)
bk[3::DOF] = ends
bk[6::DOF] = ends
bk[0::DOF] = isclose(x, L/2)
bu = ~bk

load = 50e3  # N
fext = np.zeros(N)
fext[0::DOF][isclose(x, 0)] = +load/ny
fext[0::DOF][isclose(x, L)] = -load/ny
print('resultant of the external forces:', fext.sum(), 'N')

resultant of the external forces: 0.0 N


## Solution

The linear solution at the full load, then the nonlinear one in five
equal increments of the load factor, each converged with the full
Newton-Raphson method (`NLgeom=True`); `an.static` returns the converged
load factors and the solutions of the unknown degrees of freedom.

In [5]:
callables = Callables(elements, points, weights, KC0, bu, fext)
u_lin = callables.expand(spsolve(KC0[bu][:, bu], fext[bu]))

an = Analysis(calc_fext=callables.calc_fext, calc_fint=callables.calc_fint,
              calc_kC=callables.calc_kC, calc_kG=callables.calc_kG)
an.initialInc = 0.2
an.maxInc = 0.2
increments, cs = an.static(NLgeom=True)

Started Non-Linear Static Analysis
	___________________________________________
	                                           
	Newton-Raphson solver
	___________________________________________
	Initializing...


	Step 1, attempting load factor 0.20000
		Iteration: 1, max(|R|) = 2.323e-01, zeta = 4.956e-03


		Iteration: 2, max(|R|) = 1.345e-04, zeta = 1.251e-06


		Iteration: 3, max(|R|) = 1.009e-12, zeta = 1.312e-14
		Converged at load factor 0.20000
	Step 2, attempting load factor 0.40000
		Iteration: 1, max(|R|) = 4.871e-01, zeta = 5.154e-03


		Iteration: 2, max(|R|) = 6.311e-04, zeta = 2.938e-06


		Iteration: 3, max(|R|) = 9.834e-12, zeta = 1.416e-13
		Converged at load factor 0.40000
	Step 3, attempting load factor 0.60000
		Iteration: 1, max(|R|) = 5.375e-01, zeta = 3.735e-03


... (2 outputs left out)

		Iteration: 2, max(|R|) = 1.055e-03, zeta = 2.469e-06


		Iteration: 3, max(|R|) = 3.047e-11, zeta = 2.146e-13
		Converged at load factor 0.80000
	Step 5, attempting load factor 1.00000
		Iteration: 1, max(|R|) = 6.703e-01, zeta = 2.697e-03


		Iteration: 2, max(|R|) = 1.401e-03, zeta = 2.635e-06


		Iteration: 3, max(|R|) = 5.537e-11, zeta = 3.173e-13
		Converged at load factor 1.00000
Finished Non-Linear Static Analysis
	    load factor 1.00000


In [6]:
wmax_nl = [0.] + [np.abs(callables.expand(c)[6::DOF]).max() for c in cs]
lf = [0.] + list(increments)
wmax_lin = np.abs(u_lin[6::DOF]).max()

fig, ax = plt.subplots(figsize=(5, 3.5))
ax.plot(np.array(wmax_nl)/h, np.array(lf)*load/1e3, 'o-', label='nonlinear')
ax.plot([0, wmax_lin/h], [0, load/1e3], 'k--', label='linear')
ax.set_xlabel(r'largest $|w|$ / $h$')
ax.set_ylabel('axial load (kN)')
ax.legend()
ax.grid(alpha=0.3)
plt.show()

w = callables.expand(cs[-1])[6::DOF].reshape(nx, ny)
fig, ax = plt.subplots(figsize=(7, 2.8))
c = ax.contourf(ymesh*1e3, xmesh*1e3, w/h, levels=50, cmap='coolwarm')
fig.colorbar(c, ax=ax, label='$w / h$')
ax.set_xlabel('$y$ (mm)')
ax.set_ylabel('$x$ (mm)')
ax.set_aspect('equal')
ax.set_title('radial displacement at 50 kN, nonlinear')
plt.show()

## Comparison

At this load the state is mild: the largest radial displacement is about 5 % of the
thickness, and the nonlinear solution differs from the linear one by about
5 %. The table compares the
nonlinear result with the linear one and with the value that the
docstring of the test states, 0.05 of the thickness.

In [7]:
wnl = wmax_nl[-1]/h
table([
    {'result': 'largest abs(w)/h, nonlinear', 'value': wnl, 'reference': wmax_lin/h,
     'source': 'linear solution', 'rel. diff. (%)': 100*(wnl - wmax_lin/h)/(wmax_lin/h)},
    {'result': 'largest abs(w)/h, nonlinear', 'value': wnl, 'reference': 0.05,
     'source': 'docstring of the test (two digits)', 'rel. diff. (%)': 100*(wnl - 0.05)/0.05},
])

| result | value | reference | source | rel. diff. (%) |
|---|---|---|---|---|
| largest abs(w)/h, nonlinear | 0.051144 | 0.048644 | linear solution | 5.1395 |
| largest abs(w)/h, nonlinear | 0.051144 | 0.05 | docstring of the test (two digits) | 2.2884 |

## References

- **[Crisfield1981]** Crisfield, M. A., A fast incremental/iterative solution procedure that
  handles "snap-through", Computers & Structures, 13(1-3), 55-62, 1981,
  doi:10.1016/0045-7949(81)90108-5.
- **[ArboczStarnes2002]** Arbocz, J., and Starnes, J. H., On a high-fidelity hierarchical approach
  to buckling load calculations, in New Approaches to Structural Mechanics,
  Shells and Biological Structures, Solid Mechanics and Its Applications,
  Springer Netherlands, 2002, pp. 271-292,
  doi:10.1007/978-94-015-9930-6_22.